# Traducción del Alfabeto de la Lengua de Signos en Tiempo Real

Este notebook presenta un flujo de trabajo completo para la traducción del alfabeto de la lengua de signos en tiempo real. El objetivo es desarrollar un sistema capaz de reconocer gestos de manos mediante imágenes estáticas y transmisiones de video en vivo. A continuación, se detalla cada sección del notebook.

In [ ]:
import os
os.sys
%pip cache purge
%pip uninstall -y mediapipe protobuf
%pip install opencv-python
%pip install scikit-learn
%pip install matplotlib
%pip install seaborn
%pip install xgboost
%pip install protobuf==3.20.3
%pip install mediapipe
%pip install tensorflow

Antes de comenzar con el desarrollo del proyecto, es necesario preparar el entorno de trabajo instalando las bibliotecas requeridas. Este paso asegura que contemos con todas las herramientas necesarias para procesar imágenes, realizar análisis de datos y entrenar modelos de aprendizaje automático.

En primer lugar, se limpia la caché de pip y se eliminan versiones anteriores de ciertas bibliotecas (mediapipe y protobuf) para evitar conflictos de compatibilidad. Posteriormente, se instalan las bibliotecas clave:

* OpenCV: Una biblioteca de código abierto para la visión por computadora, útil para el procesamiento de imágenes y videos.
* scikit-learn: Una biblioteca fundamental para implementar algoritmos de aprendizaje automático.
* matplotlib y seaborn: Herramientas para la visualización de datos y análisis exploratorio.
* xgboost: Un potente framework para la creación de modelos de árboles de decisión optimizados.
* protobuf: Un formato de serialización de datos utilizado por mediapipe.
* mediapipe: Una biblioteca de Google que facilita la implementación de soluciones de visión por computadora, como el seguimiento de manos.
* TensorFlow: Un framework de aprendizaje profundo para entrenar y desplegar redes neuronales.
* Estos pasos aseguran que el entorno esté listo para implementar y entrenar el modelo de reconocimiento del alfabeto en lenguaje de señas.

In [ ]:
import mediapipe as mp
import cv2
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras.utils import to_categorical

En este paso, se importan todas las bibliotecas y módulos necesarios para el desarrollo del proyecto. Estas herramientas desempeñan funciones específicas en el flujo de trabajo de la traducción del alfabeto en lenguaje de señas.

In [ ]:
MNIST_TRAIN_PATH = 'sign_mnist_train.csv'
MNIST_TEST_PATH = 'sign_mnist_test.csv'

data_mnist_train = pd.read_csv(MNIST_TRAIN_PATH)
data_mnist_test = pd.read_csv(MNIST_TEST_PATH)

x_train_mnist = np.array(data_mnist_train.drop(columns=['label']))
y_train_mnist = np.array(data_mnist_train['label'])
x_test_mnist = np.array(data_mnist_test.drop(columns=['label']))
y_test_mnist = np.array(data_mnist_test['label'])

x_train_mnist = x_train_mnist.reshape(len(x_train_mnist),28,28,1)
x_train_mnist = x_train_mnist/255.0
x_test_mnist = x_test_mnist.reshape(len(x_test_mnist),28,28,1)
x_test_mnist = x_test_mnist/255.0

model_mnist_base = tf.keras.models.Sequential()
model_mnist_base.add(tf.keras.layers.Conv2D(30, (5, 5), input_shape=(28, 28, 1), activation='relu'))
model_mnist_base.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))
model_mnist_base.add(tf.keras.layers.Conv2D(15, (3, 3), activation='relu'))
model_mnist_base.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))
model_mnist_base.add(tf.keras.layers.Dropout(0.2))
model_mnist_base.add(tf.keras.layers.Flatten())
model_mnist_base.add(tf.keras.layers.Dense(128, activation='relu'))
model_mnist_base.add(tf.keras.layers.Dense(50, activation='relu'))
model_mnist_base.add(tf.keras.layers.Dense(26, activation='softmax'))
model_mnist_base.summary()

model_mnist = model_mnist_base
model_mnist.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_mnist.fit(x_train_mnist, y_train_mnist, epochs=7, validation_split=0.2)

loss_mnist, accuracy_mnist = model_mnist.evaluate(x_test_mnist, y_test_mnist)
print(f"Loss: {loss_mnist:.4f}, Accuracy: {accuracy_mnist:.4f}")

En este paso se trabaja con el conjunto de datos Sign Language MNIST, que contiene imágenes de 28x28 píxeles representando letras del alfabeto en lenguaje de señas. El objetivo es entrenar un modelo de red neuronal convolucional (CNN) para clasificar estas imágenes de manera eficiente.

Primero, se cargan los datos desde los archivos sign_mnist_train.csv y sign_mnist_test.csv. Estos archivos contienen las etiquetas en la columna label y los valores de los píxeles en las columnas restantes. Posteriormente, los datos se dividen en características (x_train_mnist y x_test_mnist) y etiquetas (y_train_mnist y y_test_mnist).

El preprocesamiento incluye dos pasos importantes: dar forma a los datos para que coincidan con la entrada esperada por el modelo CNN (un tensor de cuatro dimensiones con las dimensiones (número de imágenes, alto, ancho, canales)) y normalizar los valores de los píxeles dividiéndolos entre 255. Esto mejora la estabilidad del entrenamiento al trabajar con valores en el rango [0, 1].

A continuación, se construye un modelo de red neuronal convolucional utilizando la API secuencial de TensorFlow. El modelo incluye capas convolucionales para extraer características espaciales de las imágenes, capas de agrupamiento para reducir la dimensionalidad, una capa de abandono para prevenir el sobreajuste, y una capa densa final con activación softmax que genera las probabilidades de cada clase. El modelo está diseñado para manejar 26 clases, correspondientes a las letras del alfabeto en lenguaje de señas (excepto la letra J).

Finalmente, el modelo se compila utilizando la función de pérdida sparse_categorical_crossentropy, adecuada para problemas de clasificación multiclase con etiquetas enteras, y el optimizador adam. El entrenamiento se realiza durante 7 épocas, utilizando el 20% del conjunto de entrenamiento para validación. Además, se genera un resumen del modelo para visualizar su arquitectura y los parámetros entrenables. Este modelo básico actúa como punto de partida para evaluar el rendimiento en la tarea de clasificación del alfabeto en lenguaje de señas.

El modelo entrenado ha alcanzado un alto rendimiento en la tarea de clasificación del alfabeto en lenguaje de señas, como lo reflejan las métricas obtenidas durante el proceso de entrenamiento y evaluación. Tiene 60,749 parámetros entrenables, lo que lo hace lo suficientemente ligero para entrenarse rápidamente sin sacrificar precisión.

Durante las 7 épocas de entrenamiento:
* La precisión en el conjunto de entrenamiento comenzó en un 28.56% y aumentó hasta un 97.95%, mientras que la pérdida disminuyó significativamente de 2.3670 a 0.0601.
* La precisión en el conjunto de validación alcanzó valores impresionantes, con un 99.96% al final del entrenamiento, acompañado de una pérdida de validación de solo 0.0043.

Estos resultados indican que el modelo ha aprendido a clasificar correctamente casi todas las imágenes en el conjunto de validación.

Después del entrenamiento, el modelo fue evaluado en un conjunto de prueba independiente, logrando:
* Una precisión de 91.93%.
* Una pérdida de 0.3197.

Aunque la precisión es ligeramente inferior a la obtenida en el conjunto de validación, sigue siendo un resultado sólido. Esta diferencia puede deberse a que el conjunto de prueba contiene datos con características ligeramente distintas o más desafiantes que los datos de entrenamiento y validación.

In [ ]:
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

hands = mp_hands.Hands(static_image_mode=True, min_detection_confidence=0.3)

DATA_PATH = './data'

data_im = []
labels_im = []
data_mp = []
labels_mp = []

for dir in os.listdir(DATA_PATH):
    for img_path in os.listdir(os.path.join(DATA_PATH, dir)):
        data_aux = []

        img = cv2.imread(os.path.join(DATA_PATH, dir, img_path))

        # Prepare dataset for CNN
        img_cnn = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        img_cnn = cv2.resize(img_cnn, (28, 28))  # Redimensionamos a 28x28
        img_cnn = np.expand_dims(img_cnn, axis=-1)  # Añadimos una dimensión para el canal
        img_cnn = img_cnn / 255.0  # Normalizamos a valores entre 0 y 1

        data_im.append(img_cnn)
        labels_im.append(str(dir))

        # Prepare dataset for Extraction + Classification
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        results = hands.process(img_rgb)
        if results.multi_hand_landmarks:
            landmarks = results.multi_hand_landmarks[0]

            # Find the bounding box of the hand
            
            x_coords = [landmark.x for landmark in landmarks.landmark]
            y_coords = [landmark.y for landmark in landmarks.landmark]

            # Normalize the landmarks
            min_x, max_x = min(x_coords), max(x_coords)
            min_y, max_y = min(y_coords), max(y_coords)

            for i in range(len(landmarks.landmark)):
                # Normalize x and y coordinates to the range [0, 1]
                normalized_x = (landmarks.landmark[i].x - min_x) / (max_x - min_x)
                normalized_y = (landmarks.landmark[i].y - min_y) / (max_y - min_y)
                data_aux.append(normalized_x)
                data_aux.append(normalized_y)

            data_mp.append(data_aux)
            labels_mp.append(str(dir))

# Convertimos las listas en arrays de numpy
data_im = np.array(data_im)
labels_im = np.array(labels_im)

data_im.shape, labels_im.shape

En esta sección, se preparan los conjuntos de datos necesarios para dos enfoques diferentes de clasificación de lenguaje de señas: uno basado en una red neuronal convolucional (CNN) y otro basado en la extracción de características de puntos clave utilizando MediaPipe. El objetivo es extraer y normalizar los datos necesarios para cada enfoque, almacenándolos en estructuras de datos separadas.

* Preparación de los datos para la CNN

    Se procesan las imágenes de las manos para crear un conjunto de datos que será utilizado por una red neuronal convolucional:
    * Cada imagen es leída desde la ruta especificada (DATA_PATH) y convertida a escala de grises mediante OpenCV.
    * Se redimensiona la imagen a 28x28 píxeles para que sea compatible con la entrada de la CNN.
    * Se normalizan los valores de los píxeles dividiéndolos entre 255, escalándolos al rango [0, 1].
    * Las imágenes procesadas se almacenan en la lista data_im, mientras que las etiquetas asociadas se guardan en labels_im.

* Preparación de los datos para la extracción de características

    Para este enfoque, se utiliza MediaPipe para extraer puntos clave (landmarks) de las manos presentes en las imágenes:
    * Las imágenes son convertidas a formato RGB para ser procesadas por MediaPipe.
    * Se aplica el modelo de detección de manos (Hands) de MediaPipe para detectar los puntos clave de la mano. Si se detecta una mano, se selecciona la primera mano encontrada.
    * Las coordenadas x e y de cada punto clave son extraídas y normalizadas dentro de un cuadro delimitador que encapsula la mano detectada:
        * Las coordenadas se ajustan para estar en el rango [0, 1], lo que garantiza que sean independientes de la resolución de la imagen.
    * Los puntos clave normalizados se almacenan en la lista data_mp, y las etiquetas correspondientes se guardan en labels_mp.

* Propósito de las listas
    * data_im y labels_im: contienen los datos procesados y las etiquetas para el enfoque basado en CNN.
    * data_mp y labels_mp: contienen las características extraídas y las etiquetas para el enfoque basado en extracción de puntos clave y clasificación tradicional.

In [ ]:
# Encode labels
label_encoder = LabelEncoder()
encoded_labels_im = label_encoder.fit_transform(labels_im)
encoded_labels_mp = label_encoder.fit_transform(labels_mp)

# Split data (using encoded_labels for stratification)
x_train_cnn, x_test_cnn, y_train_cnn, y_test_cnn = train_test_split(data_im, encoded_labels_im, test_size=0.2, shuffle=True, stratify=encoded_labels_im)
x_train_mp, x_test_mp, y_train_mp, y_test_mp = train_test_split(data_mp, encoded_labels_mp, test_size=0.2, shuffle=True, stratify=encoded_labels_mp)

# Train CNN
model_cnn = model_mnist

# Congelar todas las capas del modelo, excepto las densas
for layer in model_cnn.layers:
    if isinstance(layer, tf.keras.layers.Conv2D) or isinstance(layer, tf.keras.layers.MaxPooling2D):
        layer.trainable = False  # Congelar las capas Conv2D y MaxPooling2D
    elif isinstance(layer, tf.keras.layers.Dense):
        layer.trainable = True  # Descongelar las capas Dense

# Recompilar el modelo
model_cnn.compile(loss='sparse_categorical_crossentropy',
              optimizer=tf.keras.optimizers.Adam(learning_rate=5e-3),  # Baja tasa de aprendizaje
              metrics=['accuracy'])

# Continuar el entrenamiento
model_cnn.fit(x_train_cnn, y_train_cnn, epochs=10, validation_split=0.2)

# Train models
model_rf = RandomForestClassifier()
model_rf.fit(x_train_mp, y_train_mp)
model_lr = LogisticRegression(max_iter=1000)  # Added max_iter for convergence
model_lr.fit(x_train_mp, y_train_mp)
model_knn = KNeighborsClassifier()
model_knn.fit(x_train_mp, y_train_mp)
model_xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')  # Updated parameters for compatibility
model_xgb.fit(x_train_mp, y_train_mp)
model_dt = DecisionTreeClassifier()
model_dt.fit(x_train_mp, y_train_mp)

# Predict on test set
loss_cnn, accuracy_cnn = model_cnn.evaluate(x_test_cnn, y_test_cnn)
y_predict_cnn = model_cnn.predict(x_test_cnn)
# Convert CNN model predictions to class labels
y_predict_cnn = np.argmax(y_predict_cnn, axis=1)

y_predict_rf = model_rf.predict(x_test_mp)
y_predict_lr = model_lr.predict(x_test_mp)
y_predict_knn = model_knn.predict(x_test_mp)
y_predict_xgb = model_xgb.predict(x_test_mp)
y_predict_dt = model_dt.predict(x_test_mp)

# Accuracy score
scores = {
    "CNN": accuracy_cnn,
    "Random Forest": accuracy_score(y_test_mp, y_predict_rf),
    "Logistic Regression": accuracy_score(y_test_mp, y_predict_lr),
    "KNN": accuracy_score(y_test_mp, y_predict_knn),
    "XGBoost": accuracy_score(y_test_mp, y_predict_xgb),
    "Decision Tree": accuracy_score(y_test_mp, y_predict_dt)
}

# Plot accuracy scores as a histogram
plt.figure(figsize=(10, 6))
plt.bar(scores.keys(), scores.values(), color='skyblue')
plt.title('Model Accuracy Comparison')
plt.ylabel('Accuracy')
plt.xlabel('Models')
plt.ylim(0, 1)
plt.xticks(rotation=45)
plt.show()

# Plot confusion matrices for all models
models_predictions = {
    "CNN": y_predict_cnn,
    "Random Forest": y_predict_rf,
    "Logistic Regression": y_predict_lr,
    "KNN": y_predict_knn,
    "XGBoost": y_predict_xgb,
    "Decision Tree": y_predict_dt
}

plt.figure(figsize=(20, 10))
for i, (model_name, predictions) in enumerate(models_predictions.items(), 1):
    if model_name == "CNN":
        conf_matrix = confusion_matrix(y_test_cnn, predictions)
    else:
        conf_matrix = confusion_matrix(y_test_mp, predictions)
    plt.subplot(2, 3, i)
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
    plt.title(f'{model_name} Confusion Matrix')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')

plt.tight_layout()
plt.show()


En esta celda se lleva a cabo el entrenamiento y evaluación de diferentes modelos de clasificación. Se utilizan dos enfoques: uno basado en una red neuronal convolucional (CNN) y otro basado en métodos tradicionales de clasificación.

* Codificación de las etiquetas

    Las etiquetas de las clases (letras del alfabeto en lenguaje de señas) se codifican utilizando LabelEncoder de scikit-learn. Esto convierte las etiquetas de texto en valores numéricos que los modelos pueden procesar. Se realiza tanto para el conjunto de datos utilizado en el enfoque CNN (encoded_labels_im) como para el enfoque de extracción de características y clasificación (encoded_labels_mp).

* División de los datos
    
    Los datos se dividen en conjuntos de entrenamiento y prueba utilizando train_test_split. Esto se hace para ambos enfoques:
    * Para el modelo CNN, se utiliza el conjunto de datos de imágenes procesadas (data_im).
    * Para el modelo basado en características extraídas de las manos, se utiliza el conjunto de datos de puntos de referencia (data_mp). Se utiliza la estratificación para garantizar que la distribución de las clases en el conjunto de prueba sea representativa de la del conjunto de entrenamiento.

* Entrenamiento del modelo CNN

    El modelo CNN se congela parcialmente. Las capas convolucionales y de agrupamiento (Conv2D y MaxPooling2D) no se entrenan, mientras que las capas densas (Dense) sí se mantienen entrenables. Este enfoque se utiliza para aprovechar un modelo preentrenado y afinar solo las últimas capas. Se recompila el modelo con una tasa de aprendizaje baja (5e-3) y se entrena durante 10 épocas.

* Entrenamiento de otros modelos tradicionales

    Se entrenan varios modelos de clasificación tradicionales sobre las características extraídas de las manos:
    * Random Forest (RF): un modelo de clasificación basado en múltiples árboles de decisión.
    * Logistic Regression (LR): un modelo lineal que clasifica los datos según probabilidades.
    * K-Nearest Neighbors (KNN): un clasificador basado en la cercanía de los puntos de datos.
    * XGBoost (XGB): un clasificador basado en boosting, que utiliza árboles de decisión.
    * Decision Tree (DT): un clasificador basado en un único árbol de decisión.

* Evaluación de los modelos
    
    Se evalúan los modelos sobre el conjunto de prueba y se calculan las métricas de precisión (accuracy) para cada uno. Los resultados se almacenan en un diccionario llamado scores.

* Visualización de los resultados
    * Gráfico de barras de precisión: Se genera un gráfico de barras que compara las precisiones obtenidas por los diferentes modelos de clasificación.
    * Matriz de confusión: Se calcula y visualiza la matriz de confusión para cada modelo, lo que muestra el desempeño de cada uno en la clasificación de las clases.

Observamos que todos los modelos presentan un accuracy score superior al 90%. Sin embargo, el menor valor del accuracy es el de la red neuronal convolucional, mientras que el Random Forest que se basa en los datos extraídos con Mediapipe tiene un valor muy cercano al 100%, lo que significa que probablemente sea el más fluido sería a la hora de probar la traducción en tiempo real.

In [ ]:
# Define the augmentation ratio
augmentation_ratio = 0.5  # Adjust this as needed

# Calculate the number of augmented samples for train and test sets
num_augmented_train_mnist = int(len(x_train_mnist) * augmentation_ratio)
num_augmented_test_mnist = int(len(x_test_mnist) * augmentation_ratio)

# Select random subsets from train and test sets
train_indices = np.random.choice(len(x_train_mnist), num_augmented_train_mnist, replace=False)
x_train_mnist_subset = x_train_mnist[train_indices]
y_train_mnist_subset = y_train_mnist[train_indices]

test_indices = np.random.choice(len(x_test_mnist), num_augmented_test_mnist, replace=False)
x_test_mnist_subset = x_test_mnist[test_indices]
y_test_mnist_subset = y_test_mnist[test_indices]

# Define the data augmentation pipeline
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.1),         # Random rotation
    tf.keras.layers.RandomTranslation(0.1, 0.1), # Random translation
    tf.keras.layers.RandomZoom(0.3),             # Random zoom
    tf.keras.layers.RandomContrast(0.3),         # Random contrast
    tf.keras.layers.GaussianNoise(0.4),          # Gaussian noise
])

# Apply augmentation to the selected subsets
x_train_mnist_augmented_subset = data_augmentation(x_train_mnist_subset, training=True)
x_test_mnist_augmented_subset = data_augmentation(x_test_mnist_subset, training=True)

# Combine original data with augmented data
x_train_mnist_augmented = np.concatenate([x_train_mnist, x_train_mnist_augmented_subset])
y_train_mnist_augmented = np.concatenate([y_train_mnist, y_train_mnist_subset])
x_test_augmented = np.concatenate([x_test_mnist, x_test_mnist_augmented_subset])
y_test_augmented = np.concatenate([y_test_mnist, y_test_mnist_subset])

# Print the new shapes
x_train_mnist_augmented.shape, y_train_mnist_augmented.shape, x_test_augmented.shape, y_test_augmented.shape


In [ ]:
model_mnist_aug = model_mnist_base

model_mnist_aug.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_mnist_aug.fit(x_train_mnist_augmented, y_train_mnist_augmented, epochs=10, validation_split=0.2)

loss_aug, accuracy_aug = model_mnist_aug.evaluate(x_test_augmented, y_test_augmented)

In [ ]:
augmentation_ratio = 0.5
num_augmented = int(len(data_im) * augmentation_ratio)
# Seleccionar un subconjunto aleatorio de datos
indices = np.random.choice(len(data_im), num_augmented, replace=False)
x_subset = data_im[indices]
y_subset = labels_im[indices]
# Aplicar augmentación solo al subconjunto seleccionado
x_augmented = data_augmentation(x_subset, training=True)

# Combinar los datos originales y augmentados
data_im_augmented = np.concatenate([data_im, x_augmented])
labels_im_augmented = np.concatenate([labels_im, y_subset])

# Verificamos las dimensiones
data_im_augmented.shape, labels_im_augmented.shape

In [ ]:
encoded_labels_im_augmented = label_encoder.fit_transform(labels_im_augmented)

# Split data (using encoded_labels for stratification)
x_train_cnn_aug, x_test_cnn_aug, y_train_cnn_aug, y_test_cnn_aug = train_test_split(data_im_augmented, encoded_labels_im_augmented, test_size=0.2, shuffle=True, stratify=encoded_labels_im_augmented)

# Train CNN
model_cnn_aug = model_mnist_aug

# Congelar todas las capas del modelo, excepto las densas
for layer in model_cnn_aug.layers:
    if isinstance(layer, tf.keras.layers.Conv2D) or isinstance(layer, tf.keras.layers.MaxPooling2D):
        layer.trainable = False  # Congelar las capas Conv2D y MaxPooling2D
    elif isinstance(layer, tf.keras.layers.Dense):
        layer.trainable = True  # Descongelar las capas Dense

# Recompilar el modelo
model_cnn_aug.compile(loss='sparse_categorical_crossentropy',
              optimizer=tf.keras.optimizers.Adam(learning_rate=5e-3),  # Baja tasa de aprendizaje
              metrics=['accuracy'])

# Continuar el entrenamiento
model_cnn_aug.fit(x_train_cnn_aug, y_train_cnn_aug, epochs=10, validation_split=0.2)

loss_cnn, accuracy_cnn = model_cnn.evaluate(x_test_cnn_aug, y_test_cnn_aug)
print(f"Loss: {loss_cnn:.4f}, Accuracy: {accuracy_cnn:.4f}")

In [ ]:
# Initialize webcam capture
cap = cv2.VideoCapture(0)

# Initialize MediaPipe hands
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

hands = mp_hands.Hands(static_image_mode=False, min_detection_confidence=0.3)
margin = 20

while True:
    data_aux = []
    x_ = []
    y_ = []

    # Capture frame from webcam
    ret, frame = cap.read()

    if not ret:
        print("Failed to grab frame")
        break

    # Get frame dimensions
    H, W, _ = frame.shape

    # Convert frame to RGB for Mediapipe
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Process the frame to get hand landmarks
    results = hands.process(frame_rgb)

    # If hand landmarks are detected
    if results.multi_hand_landmarks:
        # Access the first hand's landmarks
        landmarks = results.multi_hand_landmarks[0]
        mp_drawing.draw_landmarks(
            frame,  # Image to draw on
            landmarks,  # Hand landmarks
            mp_hands.HAND_CONNECTIONS,  # Hand connections
            mp_drawing_styles.get_default_hand_landmarks_style(),
            mp_drawing_styles.get_default_hand_connections_style()
        )

        # Extract the hand landmarks and normalize them
        for i in range(len(landmarks.landmark)):
            x = landmarks.landmark[i].x
            y = landmarks.landmark[i].y

            x_.append(x)
            y_.append(y)

        # Normalize landmarks to the bounding box
        for i in range(len(landmarks.landmark)):
            x = landmarks.landmark[i].x
            y = landmarks.landmark[i].y
            data_aux.append((x - min(x_)) / (max(x_) - min(x_)))  # Normalize x to [0, 1]
            data_aux.append((y - min(y_)) / (max(y_) - min(y_)))  # Normalize y to [0, 1]

        # Make prediction
        prediction = model_rf.predict([np.asarray(data_aux)])
        predicted_character_index = prediction[0]
        predicted_character = label_encoder.inverse_transform([predicted_character_index])[0]

        # Calculate bounding box coordinates
        x1 = int(min(x_) * W) - margin
        x2 = int(max(x_) * W) + margin
        y1 = int(min(y_) * H) - margin
        y2 = int(max(y_) * H) + margin

        # Draw bounding box and predicted label on the frame
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 0), 4)
        cv2.putText(frame, predicted_character, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 1.3, (0, 0, 0), 3, cv2.LINE_AA)

    # Display the frame with the hand landmarks and predictions
    cv2.imshow('frame', frame)

    # Break the loop if 'ESC' is pressed
    if cv2.waitKey(25) == 27:  # Press 'ESC' to exit
        break

# Release the webcam and close any OpenCV windows
cap.release()
cv2.destroyAllWindows()

En esta sección se implementa un sistema en tiempo real para el reconocimiento de gestos manuales utilizando la biblioteca MediaPipe y un modelo de clasificación basado en Random Forest (RF). El sistema captura imágenes desde una cámara web mediante OpenCV, procesando cada cuadro para detectar y extraer características de las manos. Para la detección de manos, se utilizó el modelo MediaPipe Hands, configurado en modo dinámico con un umbral de confianza mínimo de 0.3 para la detección. Una vez identificadas las manos, se obtienen los 21 puntos clave de la mano, representados como coordenadas normalizadas (x, y). Estas coordenadas son normalizadas dentro de un marco delimitador que ajusta la escala y posición de los puntos clave para garantizar invariancia a traslaciones y escalas. Posteriormente, las características extraídas se organizan en un vector y se ingresan al modelo de clasificación Random Forest previamente entrenado, el cual predice la clase del gesto correspondiente. El sistema utiliza un codificador de etiquetas (Label Encoder) para mapear el índice de predicción a un carácter correspondiente, que se despliega en la interfaz gráfica. Además, se dibuja un cuadro delimitador alrededor de la mano detectada y se superpone el carácter predicho como texto en el video en tiempo real. Este proceso se repite de manera continua, logrando una inferencia en tiempo real mientras el sistema mantiene un flujo constante de cuadros desde la cámara. La implementación incluye también mecanismos para detener la ejecución mediante la tecla ESC.